# Fine-tune Whisper (small) with LoRA for Nigerian Languages 🇳🇬

A Colab-ready notebook to fine-tune **whisper-small** on **4 Nigerian languages** (Yoruba, Hausa, Igbo, Pidgin English) using LoRA via PEFT + bitsandbytes.

| Language | Dataset Config | Whisper Token |
|---|---|---|
| Yoruba | `yor_tts` | `<\|yo\|>` |
| Hausa | `hau_tts` | `<\|ha\|>` |
| Igbo | `ibo_tts` | *(not in Whisper vocab)* |
| Pidgin English | `pcm_tts` | `<\|en\|>` |

**Dataset**: [google/WaxalNLP](https://huggingface.co/datasets/google/WaxalNLP)

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets librosa evaluate jiwer accelerate bitsandbytes
!pip install -q peft

## 2. Check GPU

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU — go to Runtime > Change runtime type > GPU')
else:
    print(gpu_info)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 3. Configuration

In [ ]:
MODEL_NAME = "openai/whisper-small"
DATASET_NAME = "google/WaxalNLP"
TASK = "transcribe"
SAMPLE_RATE = 16000
OUTPUT_DIR = "whisper-small-nigerian-lora"

LANGUAGE_CONFIGS = [
    {"config": "yor_tts", "language": "yoruba"},
    {"config": "hau_tts", "language": "hausa"},
    {"config": "ibo_tts", "language": None},       # Igbo is not in Whisper's 99 languages
    {"config": "pcm_tts", "language": "english"},   # Pidgin uses English tokenizer
]

RANDOM_SEED = 42

## 4. Load Feature Extractor, Tokenizer & Processor

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, task=TASK)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, task=TASK)

## 5. Load & Preprocess Dataset

Each language is mapped **individually** so the tokenizer gets the correct language prefix token (e.g. `<|yo|>` for Yoruba). Raw audio is freed after each language to stay within Colab RAM.

In [ ]:
import gc
from datasets import load_dataset, concatenate_datasets, DatasetDict, Audio


def prepare_dataset(batch, language):
    """Prepare a single example — sets the correct language prefix token."""
    audio = batch["audio"]

    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Set language-specific prefix tokens so decoder gets the right conditioning
    if language is not None:
        tokenizer.set_prefix_tokens(language=language, task=TASK)
    else:
        tokenizer.set_prefix_tokens(task=TASK)

    batch["labels"] = tokenizer(
        batch["text"], add_special_tokens=True, truncation=True, max_length=448
    ).input_ids

    return batch


print("Loading and preprocessing datasets...")
all_train, all_val, all_test = [], [], []

for lang_cfg in LANGUAGE_CONFIGS:
    cfg, lang = lang_cfg["config"], lang_cfg["language"]
    print(f"  Processing {(lang or cfg).upper()} ({cfg})...", flush=True)

    ds = load_dataset(DATASET_NAME, cfg)
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    print(f"    train={len(ds['train']):,}  val={len(ds['validation']):,}  test={len(ds['test']):,}")

    col_names = ds["train"].column_names

    for split in ["train", "validation", "test"]:
        mapped = ds[split].map(
            prepare_dataset,
            fn_kwargs={"language": lang},
            remove_columns=col_names,
        )
        if split == "train":
            all_train.append(mapped)
        elif split == "validation":
            all_val.append(mapped)
        else:
            all_test.append(mapped)

    del ds
    gc.collect()

print("\nCombining and shuffling...")
common_voice = DatasetDict()
common_voice["train"] = concatenate_datasets(all_train + all_val).shuffle(seed=RANDOM_SEED)
common_voice["test"]  = concatenate_datasets(all_test).shuffle(seed=RANDOM_SEED)

del all_train, all_val, all_test
gc.collect()

print(f"  Train: {len(common_voice['train']):,}")
print(f"  Test : {len(common_voice['test']):,}")

## 6. Data Collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## 7. Evaluation Metrics

Text is normalized (lowercased, punctuation removed) before computing WER to avoid penalising casing/punctuation differences.

In [ ]:
import re
import evaluate

metric = evaluate.load("wer")


def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

## 8. Load Model in 8-bit & Apply LoRA

In [ ]:
from transformers import WhisperForConditionalGeneration, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, quantization_config=quantization_config, device_map="auto"
)

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [ ]:
# Needed because Whisper encoder uses Conv layers — checkpointing breaks grad flow without this
def make_inputs_require_grad(module, input, output):
    output.requires_grad_(True)

model.model.encoder.conv1.register_forward_hook(make_inputs_require_grad)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. Configure Multilingual Generation

Do **not** force a single language — let the model detect/switch languages.

In [ ]:
model.config.use_cache = False
model.config.forced_decoder_ids = None
model.generation_config.forced_decoder_ids = None
model.generation_config.language = None
model.generation_config.task = TASK

## 10. Training Arguments

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,   # effective batch = 16
    learning_rate=1e-3,
    warmup_steps=100,
    max_steps=2000,
    fp16=True,
    eval_strategy="steps", # Set evaluation strategy to 'steps' to match save strategy
    eval_steps=500,
    save_steps=500,
    logging_steps=25,
    per_device_eval_batch_size=4,
    generation_max_length=128,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,
    remove_unused_columns=False,     # required for PeftModel
    label_names=["labels"],          # required for PeftModel
    report_to=["tensorboard"],
    push_to_hub=False,
)

## 11. Trainer & Callbacks

In [ ]:
from transformers import Seq2SeqTrainer, TrainerCallback, TrainingArguments, TrainerState, TrainerControl
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR


class SavePeftModelCallback(TrainerCallback):
    """Save only the LoRA adapter weights at each checkpoint."""
    def on_save(self, args: TrainingArguments, state: TrainerState,
                control: TrainerControl, **kwargs):
        checkpoint_folder = os.path.join(
            args.output_dir, f"{PREFIX_CHECKPOINT_DIR}-{state.global_step}"
        )
        peft_model_path = os.path.join(checkpoint_folder, "adapter_model")
        kwargs["model"].save_pretrained(peft_model_path)

        pytorch_model_path = os.path.join(checkpoint_folder, "pytorch_model.bin")
        if os.path.exists(pytorch_model_path):
            os.remove(pytorch_model_path)
        return control


trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    callbacks=[SavePeftModelCallback],
)

## 12. Train! 🚀

In [ ]:
trainer.train()

## 13. Save Final Adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to: {OUTPUT_DIR}")

## 14. Evaluation

Since INT8 training doesn't support `predict_with_generate`, we run a manual eval loop with autocast.

In [ ]:
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader

eval_dataloader = DataLoader(common_voice["test"], batch_size=8, collate_fn=data_collator)

predictions = []
references = []

model.eval()
for batch in tqdm(eval_dataloader):
    with torch.amp.autocast("cuda"):
        with torch.no_grad():
            generated_tokens = (
                model.generate(
                    input_features=batch["input_features"].to("cuda"),
                    max_new_tokens=255,
                )
                .cpu()
                .numpy()
            )
            labels = batch["labels"].cpu().numpy()
            labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)

            decoded_preds = processor.tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            decoded_labels = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)

            predictions.extend([normalize_text(p) for p in decoded_preds])
            references.extend([normalize_text(l) for l in decoded_labels])

    del generated_tokens, labels, batch
    gc.collect()

# Filter empty references
valid_pairs = [(p, r) for p, r in zip(predictions, references) if r.strip()]
valid_preds, valid_refs = zip(*valid_pairs) if valid_pairs else ([], [])

wer = 100 * metric.compute(predictions=list(valid_preds), references=list(valid_refs))
print(f"\nNormalized WER on test set: {wer:.2f}%")

## 15. Inference — Load & Transcribe

To use the fine-tuned adapter later:

In [ ]:
from peft import PeftModel, PeftConfig

peft_model_id = OUTPUT_DIR  # or a HuggingFace Hub repo if you pushed it
peft_config = PeftConfig.from_pretrained(peft_model_id)

base_model = WhisperForConditionalGeneration.from_pretrained(
    peft_config.base_model_name_or_path, load_in_8bit=True, device_map="auto"
)
model = PeftModel.from_pretrained(base_model, peft_model_id)
model.config.use_cache = True

# Test with first example from test set
test_ds = load_dataset(DATASET_NAME, "yor_tts", split="test")
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

sample = test_ds[0]
input_features = feature_extractor(
    sample["audio"]["array"], sampling_rate=SAMPLE_RATE, return_tensors="pt"
).input_features.to("cuda")

with torch.amp.autocast("cuda"):
    with torch.no_grad():
        predicted_ids = model.generate(input_features, max_new_tokens=255)

transcription = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print(f"Reference : {sample['text']}")
print(f"Prediction: {transcription}")